# Minimum information tiers: metabolomics

How many detected metabolites (LC-MS features) of the Brassicaceae pilot reach each tier of the metabolite annotation, and what do the first hits look like?

**Case:** case 5, metabolomics: metabolite annotations. **Data:** annotation (evidence graphs of the pilots, `shape_constraints/rdf/` and `examples/`). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '5 metabolomics'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['annotation']
print('features with at least one annotation:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(DISTINCT ?f) AS ?n) WHERE { ?a a lodp:MetaboliteAnnotation ; lodp:feature ?f }'))
print('annotations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:MetaboliteAnnotation }'))

features with at least one annotation: 2430


annotations: 2722


## Annotation matches PlantMetWiki

`annotation-matches-plantmetwiki`

annotation-matches-plantmetwiki: A detected metabolite (feature) is annotated by at least one tool with an InChIKey whose first block (connectivity) equals that of a PlantMetWiki metabolite. The match is computed here, by comparing the first InChIKey block of each annotation with the PlantMetWiki metabolites; no stored anchor or match term is used. Returns, per feature, the number of tools whose annotation matches. OUTPUT: a list of metabolites (the PlantMetWiki metabolites that a detected metabolite is annotated as) with their putative pathways; the number of features and of annotation tools that support each one are metadata.

*Keywords:* InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, annotation tool, case 5: metabolomics, metabolite annotations, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, output: metabolites

In [3]:
r = mi.run('annotation-matches-plantmetwiki', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

rows | 2781 rows | 4.9 s


,component,componentType,pathway,supportingFeatures,supportingTools,tools
0,https://identifiers.org/inchikey/PCMORTLOPMLEFB-ONEGZZNKSA-M,metabolite,PC354,47,5,"DreaMS, SIRIUS, GNPS library search, Consensus_annotator (Annotation_matchin..."
1,https://identifiers.org/inchikey/PCMORTLOPMLEFB-ONEGZZNKSA-M,metabolite,PC190,47,5,"DreaMS, SIRIUS, GNPS library search, Consensus_annotator (Annotation_matchin..."
2,https://identifiers.org/inchikey/PCMORTLOPMLEFB-ONEGZZNKSA-M,metabolite,PC555,47,5,"DreaMS, SIRIUS, GNPS library search, Consensus_annotator (Annotation_matchin..."
3,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC446,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
4,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC340,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
5,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC220,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
6,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC236,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
7,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC172,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
8,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC272,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."
9,https://identifiers.org/inchikey/RWSXRVCMGQZWBV-WDSKDSINSA-M,metabolite,PC345,39,5,"SIRIUS, GNPS library search, MS2Query, DreaMS, Consensus_annotator (Annotati..."


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites share a chemical class

`shared-chemical-class`

shared-chemical-class: Two detected metabolites of the same dataset, each with an annotation that matches PlantMetWiki (`annotation-matches-plantmetwiki`), share a chemical class (an NPClassifier term). The class is read from the annotation as lodp:chemicalClass. NOT RUNNABLE ON THE CURRENT GRAPHS: the annotation graph has no class values yet (NPClassifier is a step of the consensus workflow, but its output is not in the RDF). OUTPUT: a list of putative metabolites of a pathway: detected metabolites that share a chemical class with a metabolite that is in the pathway (the pathway, the metabolite it is inferred from and the class are metadata).

*Keywords:* InChIKey, Minimum Information: evidence tiers, NPClassifier, PlantMetWiki metabolite, case 5: metabolomics, metabolite annotations, chemical class, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, no data yet, output: metabolites

In [4]:
r = mi.run('shared-chemical-class', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,component,componentType,pathway,viaMetabolite,sharedClass


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites linked by a similarity metric

`similarity-link`

similarity-link: Two detected metabolites are linked by a similarity metric (for example the cosine score of the MS/MS spectra, or an edge of a molecular network) above a threshold. The link is read as lodp:similarTo with lodp:similarityScore. NOT RUNNABLE ON THE CURRENT GRAPHS: no feature-to-feature similarity is in the RDF yet. OUTPUT: a list of putative metabolites of a pathway: detected metabolites linked by a similarity metric (above a threshold) to a metabolite that is in the pathway (the pathway, the metabolite it is inferred from, the metric and the score are metadata).

*Keywords:* MS/MS spectral similarity, Minimum Information: evidence tiers, case 5: metabolomics, metabolite annotations, evidence tier, local evidence graph, metabolite feature, molecular network, no data yet, output: metabolites, similarity metric

In [5]:
r = mi.run('similarity-link', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 0.0 s


,component,componentType,pathway,viaMetabolite,metric,score


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Two metabolites linked by a structure similarity (InChI)

`structure-similarity`

structure-similarity: Proxy for a structure similarity. Two detected metabolites of the same dataset are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`) and the two InChIs share the same formula layer but not the same connectivity (they are related structures: isomers). A true fingerprint similarity (for example Tanimoto) cannot be computed in SPARQL; when precomputed values are in the graph (as in `similarity-link`) they replace this proxy. OUTPUT: a list of putative metabolites of a pathway: PlantMetWiki metabolites with a related structure (here a proxy: the same InChI formula layer) to a metabolite of the pathway, both matched by detected metabolites (the pathway, the metabolite it is inferred from and the formula layer are metadata).

*Keywords:* InChI, InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, case 5: metabolomics, metabolite annotations, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, output: metabolites, structure similarity

In [6]:
r = mi.run('structure-similarity', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

timeout | None rows | None s


""


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## The difference between two metabolites is a Rhea reaction with a PlantMetWiki enzyme

`rhea-step-between-metabolites`

rhea-step-between-metabolites: Two detected metabolites are matched to PlantMetWiki metabolites (`annotation-matches-plantmetwiki`) and the difference between them is exactly one Rhea reaction: the two ChEBI compounds are on opposite sides of a Rhea reaction (federated query to Rhea), and PlantMetWiki has a reaction with that Rhea identifier and a UniProt enzyme. A hit suggests a pathway step between the two detected metabolites that PlantMetWiki may not have. Shows up to 100 pairs. OUTPUT: a list of reactions (the PlantMetWiki reactions with a Rhea reaction that connects two detected metabolites and a known UniProt enzyme) with their pathway; the Rhea reaction, the enzyme and the two detected metabolites (substrate and product features) are metadata.

*Keywords:* ChEBI, InChIKey, Minimum Information: evidence tiers, PlantMetWiki metabolite, PlantMetWiki reaction, Rhea, UniProt, case 2: inferred enzyme function from PMN/PlantCyc+ db, case 5: metabolomics, metabolite annotations, enzyme, evidence tier, federated query, local evidence graph, metabolite annotation, metabolite feature, output: reactions, pathway expansion

In [7]:
r = mi.run('rhea-step-between-metabolites', ['annotation'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

error: killed by the system (probably out of memory) | None rows | None s


""


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [8]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,annotation-matches-plantmetwiki,5 metabolomics,rows,2781.0,4.9,150,2026-11-11,r17.0.0_20260605-171052
1,rhea-step-between-metabolites,5 metabolomics,error: killed by the system (probably out of memory),NaN,NaN,150,2026-11-11,r17.0.0_20260605-171052
2,shared-chemical-class,5 metabolomics,no rows,0.0,0.0,150,2026-11-11,r17.0.0_20260605-171052
3,similarity-link,5 metabolomics,no rows,0.0,0.0,150,2026-11-11,r17.0.0_20260605-171052
4,structure-similarity,5 metabolomics,timeout,NaN,NaN,150,2026-11-11,r17.0.0_20260605-171052
